# 06 - Phát hiện Deal

Người làm: Nguyễn Huy Sơn

Đây là mục tiêu thứ 2 của dự án: so giá người bán rao với giá thị trường hợp lý của chính chiếc xe đó, rồi chia tin đăng thành 3 nhãn *Deal hời / Rẻ bất thường*, *Giá hợp lý* và *Hét giá cao*.

Đầu vào:

- `data/processed/cars_cleaned.csv`: dữ liệu sạch từ notebook 02
- `models/car_pricing_model.pkl`: mô hình định giá từ notebook 05, cho biết giá hợp lý của từng xe

Notebook này làm trước phần logic phân loại (mục 1, 2). Các phần cần giá dự đoán (mục 3 trở đi) sẽ làm khi notebook 05 có mô hình.

In [1]:
import os

CLEAN_PATH = "../data/processed/cars_cleaned.csv"
MODEL_PATH = "../models/car_pricing_model.pkl"

print("Có dữ liệu sạch:", os.path.exists(CLEAN_PATH))
print("Có mô hình từ notebook 05:", os.path.exists(MODEL_PATH))

Có dữ liệu sạch: True
Có mô hình từ notebook 05: False


## 1. Ý tưởng

Cùng một dòng xe nhưng giá rao có thể chênh nhau hàng trăm triệu (notebook 00). Một phần chênh lệch là hợp lý: xe mới hơn, đi ít hơn, bản cao hơn thì đắt hơn. Vì vậy không thể so giá một tin với giá trung bình của cả dòng xe, mà phải so với giá mô hình dự đoán cho đúng chiếc xe đó (cùng hãng, dòng xe, tuổi xe, số km, hộp số, nhiên liệu...):

**Chênh lệch (%) = (giá rao - giá dự đoán) / giá dự đoán x 100**

Chênh lệch âm nhiều là rẻ hơn thị trường, dương nhiều là đắt hơn thị trường.

## 2. Logic gắn nhãn

Một tin chỉ bị coi là rẻ hoặc đắt khi chênh lệch vượt quá một ngưỡng. Ngưỡng phải lớn hơn mức sai thông thường của mô hình, vì nếu chênh lệch nằm trong mức đó thì có thể chỉ là do mô hình đoán sai.

Nhóm tạm đặt ngưỡng 15%: với một chiếc xe có giá hợp lý 500 triệu, phải rao dưới 425 triệu mới là Deal và trên 575 triệu mới là hét giá. Mức chênh 75 triệu là đủ lớn để người mua quan tâm. Khi có mô hình từ notebook 05 sẽ kiểm tra lại: nếu mức lệch trung bình của mô hình lớn hơn 15% thì phải tăng ngưỡng.

- rẻ hơn giá dự đoán trên 15%: **Deal hời / Rẻ bất thường**
- đắt hơn giá dự đoán trên 15%: **Hét giá cao**
- còn lại: **Giá hợp lý**

Tin rẻ hơn quá nhiều chưa chắc là Deal thật, có thể là xe có vấn đề (đâm đụng, ngập nước) hoặc người bán ghi sai thông tin, nên nhãn này gọi chung là "Deal hời / Rẻ bất thường" để người mua kiểm tra kỹ trước khi mua.

In [2]:
NGUONG = 15  # %


def tinh_chenh_lech(gia_rao, gia_du_doan):
    # âm là rẻ hơn dự đoán, dương là đắt hơn
    return (gia_rao - gia_du_doan) / gia_du_doan * 100


def gan_nhan(chenh_lech):
    if chenh_lech < -NGUONG:
        return "Deal hời / Rẻ bất thường"
    elif chenh_lech > NGUONG:
        return "Hét giá cao"
    else:
        return "Giá hợp lý"

In [3]:
# thử với vài ví dụ: [giá rao, giá dự đoán] (triệu đồng)
vi_du = [[400, 500], [480, 500], [560, 500], [600, 500]]

for gia_rao, gia_du_doan in vi_du:
    chenh_lech = tinh_chenh_lech(gia_rao, gia_du_doan)
    print("Rao", gia_rao, "- dự đoán", gia_du_doan, "-> chênh", round(chenh_lech, 1), "% ->", gan_nhan(chenh_lech))

Rao 400 - dự đoán 500 -> chênh -20.0 % -> Deal hời / Rẻ bất thường
Rao 480 - dự đoán 500 -> chênh -4.0 % -> Giá hợp lý
Rao 560 - dự đoán 500 -> chênh 12.0 % -> Giá hợp lý
Rao 600 - dự đoán 500 -> chênh 20.0 % -> Hét giá cao


## 3. Gắn nhãn cho các tin đăng

*Chờ mô hình `models/car_pricing_model.pkl` từ notebook 05.*

Khi có mô hình: dùng mô hình dự đoán giá cho các tin, rồi tính chênh lệch và gắn nhãn bằng 2 hàm ở mục 2. Chỉ gắn nhãn cho những tin mô hình chưa học (tập test), vì với tin đã học thì mô hình gần như nhớ giá, chênh lệch gần bằng 0.

## 4. Thống kê theo nhãn

*Chờ mục 3.* Dự kiến: số tin và tỷ lệ theo từng nhãn, theo hãng xe và theo khu vực (TP.HCM, Hà Nội, các tỉnh khác).

## 5. Case study và phân tích sai số

*Chờ mục 3.* Dự kiến: chọn vài tin thực tế ở mỗi nhãn để kiểm tra lại, xem các tin mô hình lệch nhiều là do dữ liệu hay do mô hình.

## 6. Kịch bản định giá đầu-cuối

*Chờ mô hình từ notebook 05.* Dự kiến: nhập thông số một chiếc xe (hãng, dòng xe, năm sản xuất, số km, hộp số, tỉnh), trả về giá dự đoán, khoảng giá và nhãn Deal.

## Tóm tắt

- Đã xong: công thức chênh lệch, ngưỡng 15% và hàm gắn 3 nhãn, đã chạy thử với các ví dụ.
- Đang chờ: mô hình định giá từ notebook 05 để gắn nhãn cho các tin đăng thật, thống kê, case study và kịch bản định giá đầu-cuối.